# Model Evaluation and Diagnostics

This notebook checks the selected HistGradientBoosting regressor after the Kedro training pipeline has produced the model artefacts. The focus is on diagnostics that are easier to inspect visually here: correlation structure, prediction error by rating band, residuals, calibration, cross-validation spread, grouped-player split sensitivity, and drift reports.

The notebook rebuilds the evaluation table with the same project transformation functions used by the pipelines. It is not a separate training path.


In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit, KFold, cross_val_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from mlops_player_rating.modeling import build_model, candidate_estimators, regression_metrics, split_feature_types
from mlops_player_rating.utils import (TARGET, normalize_columns, apply_value_semantics,
    apply_attribute_imputer, fit_attribute_imputer, engineer_features, drop_non_features)

RNG = 42
BLUE, RED, GREY = "#1f77b4", "#d62728", "#7f7f7f"

def build_table():
    raw = pd.read_csv(ROOT / "data" / "01_raw" / "player_attributes.csv")
    df = normalize_columns(raw)
    df = apply_value_semantics(df)
    df = df[df[TARGET].notna()].reset_index(drop=True)
    df = apply_attribute_imputer(df, fit_attribute_imputer(df))
    df = engineer_features(df)
    groups = df["player_api_id"].copy()
    feats = drop_non_features(df, extra=[])
    return feats.drop(columns=[TARGET]), feats[TARGET].astype(float), groups

x, y, groups = build_table()
xtr, xte, ytr, yte = train_test_split(x, y, test_size=0.2, random_state=RNG, shuffle=True)
num, cat = split_feature_types(xtr)
model = build_model(candidate_estimators(RNG)["hist_gradient_boosting"], num, cat)
model.fit(xtr, ytr.to_numpy())
preds = model.predict(xte)
print("rows", len(x), "unique players", groups.nunique(), "test metrics", {k: round(v,3) for k,v in regression_metrics(yte.to_numpy(), preds).items()})

## 1. Correlation structure

The heatmap shows Pearson correlations for the target and the highest-correlation numeric fields. `potential` appears as an audit column only; it is not used for training. After excluding it, no retained single feature shows an obvious duplicate-target relationship by Pearson correlation alone.


In [ ]:
num_df = x.select_dtypes("number").copy(); num_df[TARGET] = y.to_numpy()
corr = num_df.corr(); order = corr[TARGET].abs().sort_values(ascending=False).index[:14]; c = corr.loc[order, order]
fig, ax = plt.subplots(figsize=(6.6, 5.6)); im = ax.imshow(c, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=90, fontsize=7)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=7)
for i in range(len(order)):
    for j in range(len(order)):
        ax.text(j, i, f"{c.iloc[i,j]:.2f}", ha="center", va="center", fontsize=5.5,
                color="white" if abs(c.iloc[i,j])>0.55 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("Pearson r", fontsize=8)
ax.set_title("Top features vs overall_rating"); plt.tight_layout(); plt.show()

## 2. Predictions and error by rating band

Most predictions are close to the identity line. RMSE is lowest in the dense 65-75 rating range and higher in the tails, where the sample contains fewer players.


In [ ]:
resid = yte.to_numpy() - preds
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.5))
ax[0].scatter(yte, preds, s=6, alpha=0.35, color=BLUE, edgecolors="none")
lo, hi = yte.min(), yte.max(); ax[0].plot([lo,hi],[lo,hi], color=RED, lw=1, ls="--")
ax[0].set_xlabel("actual"); ax[0].set_ylabel("predicted"); ax[0].set_title("Predicted vs actual")
bands = pd.cut(yte, bins=[0,60,65,70,75,80,100], labels=["<60","60-65","65-70","70-75","75-80",">80"])
br = {str(b): float(np.sqrt(np.mean(resid[(bands==b).to_numpy()]**2))) for b in bands.cat.categories if (bands==b).any()}
ax[1].bar(list(br), list(br.values()), color=BLUE); ax[1].set_xlabel("rating band"); ax[1].set_ylabel("RMSE"); ax[1].set_title("Error by band")
for a in ax: a.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 3. Residuals and calibration

Residuals are centered near zero, with wider spread at the low and high rating ends. The binned calibration plot compares mean predicted rating with mean observed rating; deviations are small in the dense middle of the distribution and should be interpreted more cautiously in sparse bands.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.5))
ax[0].scatter(preds, resid, s=6, alpha=0.3, color=BLUE, edgecolors="none"); ax[0].axhline(0, color=RED, lw=1, ls="--")
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("residual"); ax[0].set_title("Residuals vs predicted")
bins = np.linspace(preds.min(), preds.max(), 13); idx = np.digitize(preds, bins); xs, ys = [], []
for b in range(1, len(bins)):
    m = idx==b
    if m.sum()>5: xs.append(preds[m].mean()); ys.append(yte.to_numpy()[m].mean())
ax[1].plot([min(xs),max(xs)],[min(xs),max(xs)], color=GREY, ls="--", lw=1, label="ideal")
ax[1].plot(xs, ys, "o-", color=BLUE, ms=4, label="model"); ax[1].set_xlabel("mean predicted"); ax[1].set_ylabel("mean actual")
ax[1].set_title("Calibration"); ax[1].legend(frameon=False, fontsize=8)
for a in ax: a.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 4. Learning curve

Within the committed 6,000-row sample, held-out RMSE improves as more training rows are used and then changes less after the larger fractions. Training RMSE rises toward the held-out curve, which is consistent with reduced overfitting as sample size increases.

This plot does not rule out gains from the full 184k-row dataset. It only describes the behaviour of the sample shipped with the project.


In [ ]:
fracs = [0.1,0.2,0.35,0.5,0.7,0.85,1.0]; tr, te = [], []
for f in fracs:
    n = max(int(len(xtr)*f), 50); xs2, ys2 = xtr.iloc[:n], ytr.iloc[:n]
    m = build_model(candidate_estimators(RNG)["hist_gradient_boosting"], *split_feature_types(xs2)); m.fit(xs2, ys2.to_numpy())
    tr.append(regression_metrics(ys2.to_numpy(), m.predict(xs2))["rmse"]); te.append(regression_metrics(yte.to_numpy(), m.predict(xte))["rmse"])
sizes = [int(len(xtr)*f) for f in fracs]
fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax.plot(sizes, te, "o-", color=BLUE, label="held-out RMSE"); ax.plot(sizes, tr, "o--", color=RED, label="train RMSE")
ax.set_xlabel("training rows"); ax.set_ylabel("RMSE"); ax.set_title("Learning curve"); ax.legend(frameon=False)
ax.spines[["top","right"]].set_visible(False); plt.tight_layout(); plt.show()

## 5. Cross-validation spread

Five-fold cross-validation compares the candidate models on the training split. HistGradientBoosting has the lowest fold RMSE in this run; Random Forest is next; Ridge and Linear Regression are higher. The plot is used to check whether the selected model depends on one unusually favorable split.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RNG); names = ["hist_gradient_boosting","random_forest","ridge","linear_regression"]
labels = ["HistGB","RandomForest","Ridge","Linear"]; sc = []
for nm in names:
    mdl = build_model(candidate_estimators(RNG)[nm], num, cat)
    sc.append(-cross_val_score(mdl, xtr, ytr.to_numpy(), cv=cv, scoring="neg_root_mean_squared_error"))
fig, ax = plt.subplots(figsize=(5.6, 3.6)); bp = ax.boxplot(sc, tick_labels=labels, patch_artist=True, widths=0.6)
for p in bp["boxes"]: p.set_facecolor(BLUE); p.set_alpha(0.6)
ax.set_ylabel("5-fold CV RMSE"); ax.set_title("CV RMSE per candidate"); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 6. Random split vs. player-grouped split

The data has repeated snapshots for some players. A random split can put one player's older snapshot in training and another snapshot in test, so the notebook also evaluates a `GroupShuffleSplit` by player ID.

The grouped split is slightly worse than the random split but remains within the project thresholds. Both numbers should be reported: the random split matches the pipeline artefacts, while the grouped split is the more conservative player-overlap check.


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RNG)
tri, tei = next(gss.split(x, y, groups))
gm = build_model(candidate_estimators(RNG)["hist_gradient_boosting"], *split_feature_types(x.iloc[tri]))
gm.fit(x.iloc[tri], y.iloc[tri].to_numpy())
grp = regression_metrics(y.iloc[tei].to_numpy(), gm.predict(x.iloc[tei]))
rnd = regression_metrics(yte.to_numpy(), preds)
comp = pd.DataFrame({"random": rnd, "grouped": grp}).T[["rmse","mae","r2","within_2_pct"]].round(3)
print(comp.to_string())
fig, ax = plt.subplots(figsize=(4.6, 3.3)); xp = np.arange(2)
ax.bar(xp-0.2, [rnd["rmse"],rnd["mae"]], 0.4, color=BLUE, label="random")
ax.bar(xp+0.2, [grp["rmse"],grp["mae"]], 0.4, color=RED, label="grouped")
ax.set_xticks(xp); ax.set_xticklabels(["RMSE","MAE"]); ax.set_title("Random vs grouped split"); ax.legend(frameon=False)
ax.spines[["top","right"]].set_visible(False); plt.tight_layout(); plt.show()

## 7. Drift report: clean batch vs. simulated shift

The drift pipeline compares feature distributions with PSI and KS tests. This notebook plots the saved clean report against a simulated shifted report. The clean run has one flagged feature and does not mark dataset drift; the simulated run has multiple flagged features and crosses the configured dataset-level threshold.


In [ ]:
rep = ROOT / "data" / "08_reporting"
cf = {f["feature"]: f["psi"] for f in json.load(open(rep/"drift_report.json"))["features"]}
sf = {f["feature"]: f["psi"] for f in json.load(open(rep/"drift_report_simulated.json"))["features"]}
top = sorted(sf, key=lambda k: sf[k], reverse=True)[:14][::-1]; yp = np.arange(len(top))
fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.barh(yp+0.2, [sf[k] for k in top], 0.4, color=RED, label="simulated"); ax.barh(yp-0.2, [cf.get(k,0) for k in top], 0.4, color=BLUE, label="clean")
ax.axvline(0.1, color=GREY, ls=":", lw=1); ax.axvline(0.2, color="black", ls="--", lw=1)
ax.set_yticks(yp); ax.set_yticklabels(top, fontsize=7.5); ax.set_xlabel("PSI"); ax.set_title("Drift by feature")
ax.legend(frameon=False, loc="lower right"); ax.spines[["top","right"]].set_visible(False); plt.tight_layout(); plt.show()